In [55]:
import billboard

# SI LE PONES UN DÍA DE ENTRE-SEMANA TE DA LA CHART DE ESA SEMANA
chart1 = billboard.ChartData('hot-100', date='2026-04-01')
print(chart1)

hot-100 chart from 2026-04-04
-----------------------------
1. 'Swim' by BTS
2. 'Choosin' Texas' by Ella Langley
3. 'Man I Need' by Olivia Dean
4. 'I Just Might' by Bruno Mars
5. 'Ordinary' by Alex Warren
6. 'Golden' by HUNTR/X: EJAE, Audrey Nuna & REI AMI
7. 'So Easy (To Fall In Love)' by Olivia Dean
8. 'Stateside' by PinkPantheress With Zara Larsson
9. 'The Fate Of Ophelia' by Taylor Swift
10. 'Folded' by Kehlani
11. 'Sleepless In A Hotel Room' by Luke Combs
12. 'Opalite' by Taylor Swift
13. 'Be Her' by Ella Langley
14. 'Back To Friends' by sombr
15. 'Risk It All' by Bruno Mars
16. 'Babydoll' by Dominic Fike
17. 'Where Is My Husband!' by RAYE
18. 'Be By You' by Luke Combs
19. 'American Girls' by Harry Styles
20. 'Iloveitiloveitiloveit' by Bella Kay
21. 'E85' by Don Toliver
22. 'DTMF' by Bad Bunny
23. 'Yukon' by Justin Bieber
24. 'Dracula' by Tame Impala & JENNIE
25. 'Body To Body' by BTS
26. 'Homewrecker' by sombr
27. 'Body' by Don Toliver
28. 'Amen' by Shaboozey & Jelly Roll
29. 'Di

In [56]:
print(chart1[0:49][0])
print(chart1[8].title)
print(chart1.date)

'Swim' by BTS
The Fate Of Ophelia
2026-04-04


In [ ]:
import numpy as np

A = 1
mu = 0.5
betas = [0.2, 0.3]

class Song:
    def __init__(self, p0, t0, xs, posArr):
        self.p0 = p0
        self.t0 = t0 
        self.xs = xs
        self.posArr = posArr
        self.v0 = 'f'
    
    def S(self, A, t):
        return A/(1 + (1/self.p0 - 1)*np.e^(-t/self.t0))

    def time_step(self, newV, newPos=0):
        self.posArr.append(newPos)
        self.v0 = newV

class SongsT:
    def __init__(self, t, songs):
        self.t = t
        self.songs = songs
        self.top50 = songs[0:49]
        self.pool = songs[50:]

# def pool_maker(p, t, n=50):
#     # IN ---
#     # n = 50 (Bradlow-Fader) : # of songs added to the pool
#     # p -> prob of a song being a collab
#     # t -> week when these songs are born



In [ ]:
import billboard
import datetime as dt

def get_x1_x2_data(start_year=2013, split_year=2018, end_year=2025):
    year = start_year
    day = 1
    month = 1
    date1 = dt.date(year, month, day)
    artists_counter = {}
    x1s = {}
    collab_counter_num = 0
    x2s = {}

    while date1.year < split_year:
        chart = billboard.ChartData('hot-100', date=date1.isoformat())
        print(chart)
        for i in range(50):
            print(i)
            artist = chart.entries[i].artist
            title = chart.entries[i].title
            if artist in artists_counter.keys() and title not in x1s.keys():
                x1s[title] = artists_counter[artist]
                artists_counter[artist].append(chart.date)
            elif artist not in artists_counter.keys():
                x1s[title] = 0
                artists_counter[artist] = [chart.date]
        print(f'DONE - {date1.isoformat()}')
        date1 = date1 + dt.timedelta(days=7)

    while date1.year < end_year + 1:
        chart = billboard.ChartData('hot-100', date=date1.isoformat())
        print(chart)
        for i in range(50):
            print(i)
            artist = chart.entries[i].artist
            title = chart.entries[i].title
            if artist in artists_counter.keys() and title not in x1s.keys():
                x1s[title] = artists_counter[artist]
                artists_counter[artist].append(chart.date)
            elif artist not in artists_counter.keys():
                x1s[title] = 0
                artists_counter[artist] = [chart.date]
            if '&' in artist or ',' in artist or 'with' in artist or 'feat.' in artist or 'Feat.' in artist or 'Featuring' in artist or 'featuring' in artist or 'With' in artist and title not in x2s.keys():
                x2s[title] = 1
                collab_counter_num += 1
            elif title not in x2s.keys():
                x2s[title] = 0
        print(f'DONE - {date1.isoformat()}')
        date1 = date1 + dt.timedelta(days=7)

    return [artists_counter, collab_counter_num, x1s, x2s, date1.isoformat()]

In [ ]:
import datetime as dt

def calculate_covariates(date1, title, artist, artists_counter, collab_counter):
    hits = artists_counter(artist)
    x1 = 0
    flag = False
    i = 0
    while flag == False and x1 != len(hits):
        if dt.date.fromisoformat(hits[i]) >= date1:
            break
        else:
            x1 += 1
            i += 1
    x2 = collab_counter[title]
    return [x1, x2]
    
def get_popularity_V(t, A, p0, t0, tau_c):
    """
    Calcula V(t) usando la Ec. 4 de Soh et al.
    IMPORTANTE: t es la semana t desde que nació la canción, es decir que si tenemos un tiempo externo
    en el bucle, T, entonces t es T-{semana en la que nació la canción} 
    """
    # S(t) Logística (Ec. 2)
    def S(time):
        return A / (1 + (1/p0 - 1) * np.exp(-time/t0))
    
    # Convolución con Kernel de Memoria (Ec. 4)
    v_t = 0
    for tau in range(int(t) + 1):
        kernel = (tau + 1)**(-1) * np.exp(-tau / tau_c)
        v_t += S(t - tau) * kernel
    return v_t

# 3. FUNCIÓN DE VEROSIMILITUD (Bradlow y Fader, 2001)
def negative_log_likelihood(params, songs_data):
    """
    params: [A, p0, t0, mu, beta1, beta2]
    songs_data: Lista de diccionarios con {t_series, x1, x2}
    """
    A, p0, t0, mu, b1, b2 = params
    total_nll = 0
    
    # Evitar p0 fuera de rango [5]
    if p0 <= 0 or p0 > 1 or t0 <= 0: return 1e10
    
    for week_data in songs_data:
        # week_data contiene las canciones 'vivas' en la semana t
        worths = []
        for song in week_data['songs']:
            # Calcular tau_c específico de la canción
            tau_c = mu + b1 * song['x1'] + b2 * song['x2']
            if tau_c <= 0: tau_c = 0.01 # Restricción de positividad
            
            v_it = get_popularity_V(song['age'], A, p0, t0, tau_c)
            worths.append(v_it)
            
        # Cálculo del Exploded Logit (Ec. 4 de Bradlow & Fader)
        # Prob(ranking) = exp(V_i) / sum(exp(V_j para j >= i))
        worths = np.array(worths)
        for i in range(len(worths)):
            denom = np.sum(np.exp(worths[i:]))
            total_nll -= (worths[i] - np.log(denom))
            
    return total_nll

# 4. PROCESO DE AJUSTE 
# Inicialización de parámetros (basado en escalas típicas de Soh et al. [6])
initial_guess = [10.0, 0.1, 5.0, 0.5, 0.1, 0.1] 

# Optimización (Bradlow y Fader sugieren MCMC, pero MLE es más rápido para empezar)
# result = minimize(negative_log_likelihood, initial_guess, args=(processed_data,))

In [38]:
import datetime as dt
date1 = dt.date(1989, 1, 13)
td = dt.timedelta(days=7)
print(date1 + td)

1989-01-20


In [ ]:
import csv
import numpy as np

[artists_counter, collab_counter_num, x1s, x2s, end_date] = get_x1_x2_data()

with open('artists_counter.csv', 'w') as csv_file:  
    writer = csv.writer(csv_file)
    for key, value in artists_counter.items():
       writer.writerow([key, value])

with open('x1s.csv', 'w') as csv_file:  
    writer = csv.writer(csv_file)
    for key, value in x1s.items():
       writer.writerow([key, value])

with open('x2s.csv', 'w') as csv_file:  
    writer = csv.writer(csv_file)
    for key, value in x2s.items():
       writer.writerow([key, value])

start_date = '2018-01-01'
collab_num = np.array([collab_counter_num, start_date, end_date])
np.save('collab_num', collab_num)

hot-100 chart from 2013-01-05
-----------------------------
1. 'Locked Out Of Heaven' by Bruno Mars
2. 'Diamonds' by Rihanna
3. 'Ho Hey' by The Lumineers
4. 'I Knew You Were Trouble.' by Taylor Swift
5. 'Beauty And A Beat' by Justin Bieber Featuring Nicki Minaj
6. 'Die Young' by Ke$ha
7. 'One More Night' by Maroon 5
8. 'I Cry' by Flo Rida
9. 'Home' by Phillip Phillips
10. 'Thrift Shop' by Macklemore & Ryan Lewis Featuring Wanz
11. 'Don't You Worry Child' by Swedish House Mafia Featuring John Martin
12. 'Scream & Shout' by will.i.am & Britney Spears
13. 'Try' by P!nk
14. 'Some Nights' by fun.
15. 'Let Me Love You (Until You Learn To Love Yourself)' by Ne-Yo
16. 'Girl On Fire' by Alicia Keys Featuring Nicki Minaj
17. 'The A Team' by Ed Sheeran
18. 'It's Time' by Imagine Dragons
19. 'Gangnam Style' by PSY
20. 'Swimming Pools (Drank)' by Kendrick Lamar
21. 'All I Want For Christmas Is You' by Mariah Carey
22. 'Cruise' by Florida Georgia Line Featuring Nelly
23. 'Clique' by Kanye West, Jay-

KeyboardInterrupt: 